# ViMedNer — Medical NER notebook
This notebook implements the training / evaluation pipeline from the repository README and wires together the Python modules in this workspace for quick interactive use.

## Setup
Run the next cell to ensure required packages are installed (uses the repository `requirements.txt`).

In [1]:
%pip install -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


## Imports and workspace setup

In [2]:
import os
import sys
from pathlib import Path
# Ensure notebook runs from the repository package folder
repo_dir = Path('.').resolve()
os.chdir(repo_dir)
sys.path.append(str(repo_dir))
print('CWD:', repo_dir)
# Standard imports used by the repo code
import logging
import numpy as np
from transformers import AutoConfig, AutoTokenizer, AutoModelForTokenClassification, TrainingArguments, Trainer, set_seed
from seqeval.metrics import classification_report
# Local modules
from utils_ner import TokenClassificationDataset, Split
from tasks import NER
try:
    from model import BertCRF, BertLstmCRF
except Exception as e:
    # custom models may require additional deps; fall back to HF model if import fails
    BertCRF = None
    BertLstmCRF = None
print('Imports OK')

CWD: C:\Users\NCPC\OneDrive\Desktop\Lab1\NLP\ViMedNer


c:\Users\NCPC\miniconda3\envs\myenv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Imports OK


## Data: labels and quick peek

In [3]:
# Use the task class to load labels and count examples
task = NER()
labels = task.get_labels('data/labels.txt') if os.path.exists('data/labels.txt') else task.get_labels(None)
print('Labels (count):', len(labels))
# Read a few examples from the train file to inspect format
examples = task.read_examples_from_file('data', Split.train) if os.path.exists('data/train.txt') else []
print('Example sentences loaded:', len(examples))
if len(examples) > 0:
    ex = examples[0]
    print('GUID:', ex.guid)
    print('Words:', ex.words[:50])
    print('Labels:', ex.labels[:50])

Labels (count): 11
Example sentences loaded: 4573
GUID: train-1
Words: ['các', 'khối', 'u', 'não', 'xảy', 'ra', 'thường', 'xuyên', 'hơn', 'ở', 'người', 'da', 'trắng', 'hơn', 'là', 'ở', 'người', 'của', 'các', 'chủng', 'tộc', 'khác', '.']
Labels: ['O', 'B-ten_benh', 'I-ten_benh', 'I-ten_benh', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']


## Tokenizer, config, and model setup

In [4]:
# Choose a model name (use a model from README or HuggingFace). Change here as needed.
MODEL_NAME = 'distilbert-base-uncased'
num_labels = len(labels)
config = AutoConfig.from_pretrained(MODEL_NAME, num_labels=num_labels, id2label={i: l for i, l in enumerate(labels)}, label2id={l: i for i, l in enumerate(labels)})
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
# Load either a custom CRF model if requested & available, otherwise HF token classification model
use_crf = False
use_lstmcrf = False
if use_crf and BertCRF is not None:
    model = BertCRF.from_pretrained(MODEL_NAME, config=config)
elif use_lstmcrf and BertLstmCRF is not None:
    model = BertLstmCRF.from_pretrained(MODEL_NAME, config=config)
else:
    model = AutoModelForTokenClassification.from_pretrained(MODEL_NAME, config=config)
print('Model:', type(model))

c:\Users\NCPC\miniconda3\envs\myenv\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of DistilBertForTokenClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model: <class 'transformers.models.distilbert.modeling_distilbert.DistilBertForTokenClassification'>


## Prepare datasets using the repository dataset class

In [5]:
max_seq_length = 128
train_dataset = None
eval_dataset = None
if os.path.exists('data'):
    train_dataset = TokenClassificationDataset(token_classification_task=task, data_dir='data', tokenizer=tokenizer, labels=labels, model_type=config.model_type, n_obs=-1, max_seq_length=max_seq_length, overwrite_cache=True, mode=Split.train) if len(examples) > 0 else None
    eval_dataset = TokenClassificationDataset(token_classification_task=task, data_dir='data', tokenizer=tokenizer, labels=labels, model_type=config.model_type, max_seq_length=max_seq_length, overwrite_cache=True, mode=Split.dev) if os.path.exists('data/dev.txt') else None
    print('Train dataset:', len(train_dataset) if train_dataset is not None else 0, 'Eval dataset:', len(eval_dataset) if eval_dataset is not None else 0)
else:
    print('data/ directory not found in repository root; please add the dataset files as specified in README')

Train dataset: 4573 Eval dataset: 1524


## Prediction alignment and evaluation helper functions

In [6]:
def align_predictions(predictions: np.ndarray, label_ids: np.ndarray, label_map):
    if len(predictions.shape) == 3:
        preds = np.argmax(predictions, axis=2)
    else:
        preds = predictions
    batch_size, seq_len = preds.shape
    out_label_list = [[] for _ in range(batch_size)]
    preds_list = [[] for _ in range(batch_size)]
    for i in range(batch_size):
        for j in range(seq_len):
            if label_ids[i, j] != -100:
                preds_list[i].append(label_map[preds[i][j]])
                out_label_list[i].append(label_map[label_ids[i][j]])
    return preds_list, out_label_list

def compute_metrics(p, label_map):
    preds_list, out_label_list = align_predictions(p.predictions, p.label_ids, label_map)
    return {
        'classification_report': classification_report(out_label_list, preds_list, digits=3)
    }

label_map = {i: l for i, l in enumerate(labels)}

## Training with Hugging Face Trainer (example)

In [7]:
# Small example training run configuration — adjust args for real training
output_dir = './output'
training_args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=1,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    logging_strategy='epoch',
    save_strategy='no',
    evaluation_strategy='no',
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset if train_dataset is not None else None,
    eval_dataset=eval_dataset if eval_dataset is not None else None,
    tokenizer=tokenizer,
)

# Run training if data is present
if train_dataset is not None and len(train_dataset) > 0:
    trainer.train()
else:
    print('No training data found; populate data/ and re-run the cell to train.')

c:\Users\NCPC\miniconda3\envs\myenv\lib\site-packages\transformers\training_args.py:1474: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
  0%|          | 0/286 [00:00<?, ?it/s]

AttributeError: 'list' object has no attribute 'keys'

## Evaluation / Prediction example

In [ ]:
# Example: run evaluation or predict on the test split
if eval_dataset is not None and len(eval_dataset) > 0:
    res = trainer.evaluate()
    print(res)
else:
    print('No eval dataset present; create data/dev.txt to evaluate.')

if os.path.exists('data/test.txt'):
    test_dataset = TokenClassificationDataset(token_classification_task=task, data_dir='data', tokenizer=tokenizer, labels=labels, model_type=config.model_type, max_seq_length=max_seq_length, overwrite_cache=True, mode=Split.test)
    preds_output = trainer.predict(test_dataset)
    print('Prediction completed')
else:
    print('No test set found at data/test.txt')

## Notes & next steps
- Update `MODEL_NAME` to a model from the README (e.g. `vinai/phobert-base`).
- Adjust `TrainingArguments` for full training runs and GPU usage.
- If you want to use the repo's CRF/LSTM-CRF models, set `use_crf` or `use_lstmcrf` to True and ensure `model.py` dependencies are installed.
- To run training as in the README, either use this notebook or run `python run_ner.py` with the provided CLI arguments.